<a href="https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
!git clone https://github.com/Debbie1236-cmd/flyrank-ml-internship.git
%cd flyrank-ml-internship/work/notebooks

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 234 (delta 110), reused 100 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (234/234), 2.14 MiB | 25.79 MiB/s, done.
Resolving deltas: 100% (110/110), done.
/content/flyrank-ml-internship/work/notebooks/flyrank-ml-internship/work/notebooks/flyrank-ml-internship/work/notebooks/flyrank-ml-internship/work/notebooks


In [19]:
%pip -q install duckdb huggingface_hub

In [20]:
import os
import getpass

# Try environment variable first
HF_TOKEN = os.environ.get("HF_TOKEN")

# Then try Colab Secrets
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

# Last resort: ask you to paste it
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

In [21]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [22]:
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    # ...more tables below this
}

In [23]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## Unit of Analysis

One row represents the daily search performance of a single content page (content_hash_id) for a single client (client_hash_id) on one reporting date (report_date). I selected March 2026 as the analysis window because it is a mid-panel month and avoids using the final month of the dataset.

Verification: A query on the fact_daily table for March 2026 returned 9,841,378 rows, with dates ranging from 2026-03-01 to 2026-03-31, confirming the selected analysis window.

In [24]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_data_available

Label (Proxy):
- Whether a content page's search impressions decline over time. This serves as a proxy label that can later be derived from historical search performance.

Context:
- report_date
- client_hash_id
- content_hash_id

Excluded:
- month (excluded because it is derived from report_date and does not provide additional information.)

In [25]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_data_available,
    month
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_data_available,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,True,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,True,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,True,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,True,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,True,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
# Query 1: Verify the grain
print("Query 1: Duplicate check")
display(con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df())

# Query 2: Row count and date range
print("Query 2: Row count and date range")
display(con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df())

# Query 3: Availability
print("Query 3: GSC availability")
display(con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
""").df())

Query 1: Duplicate check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Query 2: Row count and date range


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


Query 3: GSC availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


Query 1 – Grain

A query checking for duplicate combinations of report_date, client_hash_id, and content_hash_id returned no rows, confirming that each row represents one content page for one client on one report date.

Query 2 – Row count and time window

A query for March 2026 returned 9,841,378 rows, with dates ranging from 2026-03-01 to 2026-03-31, confirming the selected analysis window.

Query 3 – Availability

Filtering with gsc_data_available IS TRUE returned 3,611,061 rows, confirming that analyses based on Google Search Console metrics are limited to records where GSC data is available.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [27]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
""").df()

,available_rows
0,3611061


Data limitation:
Not all records contain Google Search Console data, and the dataset has an unbalanced history because different clients began collecting data at different times. As a result, analyses based on March 2026 and GSC metrics apply only to records where gsc_data_available is TRUE.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.